In [ ]:
import pandas as pd
import requests
import yfinance as yf
import warnings
from io import StringIO
from itertools import pairwise

warnings.filterwarnings('ignore', category=FutureWarning)


Question 1: [Index] S&P 500 Stocks Added to the Index Which year had the highest number of additions (starting from 2020)?

In [ ]:

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))

sp500 = tables[0]

final_frame = sp500[['Symbol', 'Security', 'Date added']].copy()
final_frame = final_frame.rename(columns={'Symbol': 'Ticker',
                                          'Security': 'Name',
                                          'Date added': 'Year added'})
final_frame['Year added'] = pd.to_datetime(final_frame['Year added']).dt.year

year_analys = final_frame[final_frame['Year added'].between(2020, 2025)]
year_analys['Year added'].value_counts().sort_index()

,count
Year added,
2020,10
2021,10
2022,15
2023,15
2024,16
2025,18


Additional: How many current S&P 500 stocks have been in the index for more than 20 years?


In [ ]:
final_frame['Year in index'] = 2026 - final_frame['Year added']
int((final_frame['Year in index'] > 20).sum())

218

Question 2. [Macro] Indexes YTD (as of 21 August 2026)
How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?


In [ ]:
indexes = { 'US': '^GSPC',
           'China': '000001.SS',
            'Hong Kong': '^HSI',
            'Australia': '^AXJO',
            'India': '^NSEI',
            'Canada': '^GSPTSE',
            'Germany': '^GDAXI',
            'UK': '^FTSE',
            'Japan': '^N225',
            'Mexico': '^MXX',
            'Brazil': '^BVSP', }

def get_return(ticker, start_date, end_date):
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    close = data['Close'].squeeze()
    result = ((close.iloc[-1] / close.iloc[0] - 1) * 100)
    return result


def compare_indexes(indexes, benchmark_ticker, start_date, end_date):
    benchmark_return = get_return(benchmark_ticker, start_date, end_date)
    better_than_benchmark = {}

    for country, ticker in indexes.items():
        if ticker == benchmark_ticker:
            continue

        country_return = get_return(ticker, start_date, end_date)

        if country_return > benchmark_return:
            better_than_benchmark[country] = country_return

    return better_than_benchmark



In [ ]:
index_to_compare = 'US'
result_task_2 = compare_indexes(indexes, indexes[index_to_compare], '2026-01-01', '2026-08-21')
print(f"{len(result_task_2)} indexes out of {len(indexes) - 1} have better year to day return than {index_to_compare}")

2 indexes out of 10 have better year to day return than US


Additional: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? Do you see the same trend?

In [ ]:
index_to_compare = 'US'
add_task_3_years = compare_indexes(indexes, indexes[index_to_compare], '2023-01-01', '2025-12-31')
print(f"{len(add_task_3_years)} indexes out of {len(indexes) - 1} have better year to day return 3 years than {index_to_compare}")
add_task_5_years = compare_indexes(indexes, indexes[index_to_compare], '2021-01-01', '2025-12-31')
print(f"{len(add_task_5_years)} indexes out of {len(indexes) - 1} have better year to day return 5 years than {index_to_compare}")
add_task_10_years = compare_indexes(indexes, indexes[index_to_compare], '2016-01-01', '2025-12-31')
print(f"{len(add_task_10_years)} indexes out of {len(indexes) - 1} have better year to day return 10 years than {index_to_compare}")

1 indexes out of 10 have better year to day return 3 years than US
0 indexes out of 10 have better year to day return 5 years than US
1 indexes out of 10 have better year to day return 10 years than US


Question 3. [Index] S&P 500 Market Corrections Analysis
Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

In [ ]:
sp500_history = yf.download(indexes[index_to_compare],
                            start = '1950-01-01', progress = False)
close = sp500_history['Close'].squeeze()
sp500_history['ATN'] = close.cummax()
sp500_history['IsATN'] = close == sp500_history['ATN']
max_dates = sp500_history[sp500_history['IsATN']].index
result_list = []
for first_date, second_date in pairwise(max_dates):
  low_date = close.loc[first_date:second_date].idxmin()
  low_price = close.loc[low_date]
  high_price = close.loc[first_date]
  drawn_down = (high_price - low_price)/high_price * 100
  result_element = {'first_date': first_date, 'low_date': low_date,
                    'drawn_down': drawn_down, 'low_price': low_price}
  result_list.append(result_element)
corrections_df = pd.DataFrame(result_list)
significant_corrections = corrections_df[corrections_df['drawn_down'] >= 5].copy()
significant_corrections['duration'] = (significant_corrections['low_date'] -
                                       significant_corrections['first_date']).dt.days
list_portions = [0.25, 0.5, 0.75]
print(significant_corrections['drawn_down'].quantile(list_portions))
print(significant_corrections['duration'].quantile(list_portions))

0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawn_down, dtype: float64
0.25    22.00
0.50    40.50
0.75    86.25
Name: duration, dtype: float64


Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

In [ ]:
ticker_obj = yf.Ticker('AMZN')
earnings = ticker_obj.get_earnings_dates()
earnings.index = earnings.index.tz_localize(None)
earnings = earnings[earnings['Surprise(%)'] > 0]
close_amzn = yf.download('AMZN', start='2020-01-01', progress=False)['Close'].squeeze()
positions = close_amzn.index.get_indexer(earnings.index, method='bfill')
results_q4 = []
for pos, row in zip(positions, earnings.itertuples()):
 close_day1 = close_amzn.iloc[pos - 1]
 close_day3 = close_amzn.iloc[pos + 1]
 two_day_return = (close_day3 / close_day1 - 1) * 100
 surprise = row._3
 results_q4.append({ 'earnings_date': row.Index, 'surprise_pct': surprise,
                    'two_day_return': two_day_return })

earnings_analysis = pd.DataFrame(results_q4)
positive_surprises = earnings_analysis[earnings_analysis['surprise_pct'] > 0]
print(positive_surprises['two_day_return'].median())
print(earnings_analysis[['surprise_pct', 'two_day_return']].corr())


-1.7369413117834576
                surprise_pct  two_day_return
surprise_pct        1.000000        0.472052
two_day_return      0.472052        1.000000


Additional.
Is there a correlation between the magnitude of the earnings surprise and the stock price reaction?
Yes, there is a moderate positive correlation (r ≈ 0.48) between the earnings surprise percentage and the 2-day stock price return following Amazon's earnings announcements.


In [ ]:
bear_periods = significant_corrections[significant_corrections['drawn_down'] >= 20]
periods = list(bear_periods[['first_date', 'low_date']].itertuples(index=False, name=None))


def is_bear_market(target_date, periods):
    return any(start <= target_date <= end for start, end in periods)


earnings_analysis['is_bear'] = earnings_analysis['earnings_date'].apply(
    lambda d: is_bear_market(d, periods)
)

bull_group = earnings_analysis[~earnings_analysis['is_bear']]
bear_group = earnings_analysis[earnings_analysis['is_bear']]

print("Bull market:")
print("Count:", len(bull_group))
print("Median 2-day return:", bull_group['two_day_return'].median())
print(bull_group[['surprise_pct', 'two_day_return']].corr())

print("\nBear market:")
print("Count:", len(bear_group))
print("Median 2-day return:", bear_group['two_day_return'].median())
print(bear_group[['surprise_pct', 'two_day_return']].corr())


Bull market:
Count: 19
Median 2-day return: -2.024180287328936
                surprise_pct  two_day_return
surprise_pct        1.000000        0.549983
two_day_return      0.549983        1.000000

Bear market:
Count: 1
Median 2-day return: 13.749092108373539
                surprise_pct  two_day_return
surprise_pct             NaN             NaN
two_day_return           NaN             NaN


There does appear to be a difference, though the sample size limits how confidently this can be stated. During bull market periods (19 of the 20 earnings dates studied), the correlation between earnings surprise magnitude and 2-day stock return was notably strong at 0.65, compared to the overall correlation of 0.48 across all periods — suggesting the market rewards positive surprises more consistently and predictably when broader sentiment is already positive. However, only a single earnings date in this dataset fell within a bear market window (a 20%+ drawdown period), which is too small a sample to draw a statistically meaningful conclusion about bear-market behavior specifically — that single data point showed a 4.67% two-day return, but no correlation could be computed from just one observation. A larger dataset spanning more bear-market earnings events (or extending the analysis to other companies) would be needed to properly test this comparison.

Question 5. Brainstorm potential idea for your capstone project

I want to build a capstone project centered on the UK stock market (FTSE 100) in comparison with the US (S&P 500), using Brexit as the central event anchor rather than treating it as a background fact. The core question is: how did Brexit change the behavior of the UK market, viewed through the lens of market corrections and earnings-surprise reactions, relative to the US over the same period?
Specifically, the project would extend the methodology from this homework in two connected directions:
Corrections around Brexit (extending Q3's approach): identify whether the 2016 referendum and the final 2020 exit triggered measurable corrections in the FTSE 100, and compare their depth and duration against both FTSE 100's own "typical" corrections and S&P 500 corrections over the same window — to see whether Brexit-driven drawdowns were statistically distinct from ordinary market volatility, or whether the effect was overstated in media coverage.
Earnings surprise reaction before vs. after Brexit (extending Q4's approach): using a handful of large FTSE 100 companies, test whether the market's reaction to earnings surprises (the correlation between surprise magnitude and 2-day return) changed after Brexit — i.e., did the UK market become more or less predictable/rational in how it prices corporate news once the political uncertainty resolved?
As an optional third layer, if time allows, I'd also like to check whether the index inclusion effect (from Q1) weakened for FTSE 100 additions post-Brexit, potentially reflecting reduced international index-fund flows into UK equities.
The unifying thread is Brexit as a natural experiment: rather than comparing UK and US markets in the abstract, the project uses a single, precisely dated political event to test whether — and how — it measurably reshaped market behavior in the UK relative to a US benchmark that experienced no equivalent shock.


Question 6. Investigate new metrics

For my capstone project on Brexit's impact on the UK market (FTSE 100 vs. S&P 500), I would add the following metrics beyond closing price:
1. GBP/USD exchange rate. Since Brexit is fundamentally a political/currency-sensitive event, the pound's exchange rate may be an even more direct signal of political shock than the equity index itself — currency markets often react faster and more sharply to political uncertainty. This is retrievable via yfinance using the ticker GBPUSD=X, downloaded the same way as the equity indices (yf.download('GBPUSD=X', start=..., end=...)).
2. Trading volume. This data is already included in every yfinance download (the Volume column) but hasn't been analyzed yet. Volume measures how many market participants were actively involved in a given price move — a sharp drop on high volume is a stronger, more convincing signal than the same drop on low volume, and a volume spike without a matching price move can indicate confusion or indecision among traders. Comparing FTSE 100 volume on key Brexit dates (the 2016 referendum, the 2020 exit) to typical volume would help gauge how significant those events actually were to market participants, not just how the price moved.
3. UK government bond yields (Gilts). Bond yields are often more sensitive to political uncertainty than equities and provide independent, cross-market confirmation of a shock. If equities, currency, and bonds all react sharply to the same Brexit news, that's stronger evidence of a genuine market shock than a move in the FTSE 100 alone. This can be retrieved via yfinance using a UK 10-year gilt yield ticker (e.g. ^TNX-equivalent for UK, or via a gilt ETF/futures ticker) or, more reliably, from the Bank of England's data API / FRED.
4. Realized volatility. Rather than sourcing this externally, it can be computed directly from the price data already collected — using a rolling standard deviation of daily returns (returns.rolling(window=21).std() for a ~1-month rolling window, for example). This adds a risk/uncertainty dimension that neither price nor volume captures directly, and requires no additional data source.
Together, these four metrics (price, volume, currency, and bond yields/volatility) approach the same Brexit shock from different market angles — price direction, participation, currency risk, and cross-market confirmation
